# 02 — Linking every player to his Transfermarkt card

Goal: give each FPL player (season, element) his Transfermarkt player_id. First the clubs and seasons, then the player names which to be normalized

In [1]:
import sys
sys.path.append("..")

from rapidfuzz import fuzz
from unidecode import unidecode
import numpy as np
import pandas as pd
from src.load import load_gw, load_valuations, load_players, load_fpl_players

gw = load_gw()
val = load_valuations()
players = load_players()
fpl_players = load_fpl_players()

## Step 1 — Premier League clubs per season

In [2]:
pl_clubs = gw[["season", "team"]].drop_duplicates()
assert len(pl_clubs) == 100  # 5 seasons x 20 clubs
pl_clubs.head()

,season,team
0,2021-22,Man Utd
1,2021-22,Aston Villa
2,2021-22,Watford
3,2021-22,Southampton
5,2021-22,Liverpool


## Step 2 — Club name mapping (Transfermarkt to FPL)

In [3]:
club_map = {
    "Arsenal FC": "Arsenal",
    "Aston Villa": "Aston Villa",
    "AFC Bournemouth": "Bournemouth",
    "Brentford FC": "Brentford",
    "Brighton & Hove Albion": "Brighton",
    "Burnley FC": "Burnley",
    "Chelsea FC": "Chelsea",
    "Crystal Palace": "Crystal Palace",
    "Everton FC": "Everton",
    "Fulham FC": "Fulham",
    "Ipswich Town": "Ipswich",
    "Leeds United": "Leeds",
    "Leicester City": "Leicester",
    "Liverpool FC": "Liverpool",
    "Luton Town": "Luton",
    "Manchester City": "Man City",
    "Manchester United": "Man Utd",
    "Newcastle United": "Newcastle",
    "Norwich City": "Norwich",
    "Nottingham Forest": "Nott'm Forest",
    "Sheffield United": "Sheffield Utd",
    "Southampton FC": "Southampton",
    "Sunderland AFC": "Sunderland",
    "Tottenham Hotspur": "Spurs",
    "Watford FC": "Watford",
    "West Ham United": "West Ham",
    "Wolverhampton Wanderers": "Wolves",
}

assert set(club_map.values()) == set(gw["team"].unique())  # every FPL team is mapped

assert set(club_map.keys()) - set(val["current_club_name"].unique()) == set()  # every name in the dict exists in val 

## Step 3 — Season of each valuation

I chose July 1st as the boundary: the season usually ends in May, so June valuations are basically a recap of the season that just finished, and the player is still listed at his old club.

Unfortunately a deal done in late June or a value posted early July can land on the wrong side, but those cases should show up as unmatched and I'll check them by hand.

In [4]:
year = val["date"].dt.year
month = val["date"].dt.month

# season starts this year if month >= 7, otherwise it started last year
start_year = np.where(month >= 7, year, year - 1)

# build "2022-23" from start_year = 2022
start = pd.Series(start_year, index=val.index).astype(str)
end = (pd.Series(start_year, index=val.index) + 1).astype(str).str[2:4]
val["season"] = start + "-" + end

In [5]:
# check the July 1st boundary on a known transfer (Rice: West Ham -> Arsenal, July 2023)
rice_id = players[players["name"] == "Declan Rice"]["player_id"].iloc[0]
val[val["player_id"] == rice_id][["date", "current_club_name", "season"]]

,date,current_club_name,season
803,2021-07-15,West Ham United,2021-22
16230,2021-12-23,West Ham United,2021-22
46896,2022-06-15,West Ham United,2021-22
66310,2022-11-03,West Ham United,2022-23
104045,2023-06-20,West Ham United,2022-23
120794,2023-10-09,Arsenal FC,2023-24
133144,2023-12-19,Arsenal FC,2023-24
148784,2024-05-27,Arsenal FC,2023-24
178933,2024-12-16,Arsenal FC,2024-25
195158,2025-05-30,Arsenal FC,2024-25


## Step 4 — Keep only Premier League valuations

Transfermarkt's competition column can't be trusted (see 01 and the conclusion), so I don't use it: I keep a valuation only if its club was in the Premier League that season.

In [6]:
# Transfermarkt name -> FPL name, NaN if not one of the 27 PL clubs
val["team"] = val["current_club_name"].map(club_map)

# keep a valuation only if its (season, team) pair is in the Premier League that season
val_pl = val.merge(pl_clubs, on=["season", "team"], how="inner")

print(len(val), len(val_pl))
val_pl["season"].value_counts().sort_index()

253016 7068


season
2021-22    1250
2022-23    1496
2023-24    1401
2024-25    1389
2025-26    1532
Name: count, dtype: int64

Testing on a club that got relegated (Leeds, in the Championship in 2023-24 and 2024-25): in val_pl we only get the seasons where Leeds was in the PL.

In [7]:
print(val[val["team"] == "Leeds"]["season"].value_counts().sort_index())
print(val_pl[val_pl["team"] == "Leeds"]["season"].value_counts().sort_index())

season
2021-22    57
2022-23    79
2023-24    60
2024-25    53
2025-26    81
Name: count, dtype: int64
season
2021-22    57
2022-23    79
2025-26    81
Name: count, dtype: int64


## Step 5 — Matching player names

As I noticed at the end of the first notebook, names are not written the same way in both datasets. FPL often uses the full legal name ("Bruno Miguel Borges Fernandes") while Transfermarkt uses the usual name ("Bruno Fernandes"). There are also accents (Ødegaard), word order (Son Heung-min vs Heung-min Son) and nicknames (Rodri, Alisson).

What I decided:
- I only compare names inside the same club and season, so there are only like 30 candidates instead of 50k players, and fewer false matches (for example two players who share the same name).
- I normalize names first (no accents, lowercase) so Ødegaard and Odegaard are the same.
- For nicknames I can use FPL's `web_name` (short name shown in the game), which is easier to handle tbh.

I'll use `unidecode` to normalize and `rapidfuzz` to match. I went through the rapidfuzz docs to see which scorer fits best: `token_sort_ratio` handles word order, but `token_set_ratio` handles both word order and extra middle names, so I'll use that one. I didn't take `partial_token_set_ratio` because it gives 100 as soon as one word is shared.

In [8]:
def normalize(name):
    name = unidecode(name)          # remove accents
    name = name.lower()             # lowercase
    name = name.strip()             # remove spaces at the start and end
    name = name.replace("-", " ")   # "heung-min" -> "heung min"
    return name

print(normalize("Martin Ødegaard"))
print(normalize("Heung-Min Son"))
print(normalize("  Bruno Fernandes "))

martin odegaard
heung min son
bruno fernandes


Preparing the two tables to compare: one row per player, season and club on each side.

In [9]:
# one row per player, season and club
fpl_side = gw[["season", "element", "team", "name"]].drop_duplicates()

# add web_name from players_raw to have both in case one is better for matching
fpl_side = fpl_side.merge(
    fpl_players[["season", "element", "web_name"]], on=["season", "element"], how="left"
)

# normalized names
fpl_side["name_norm"] = fpl_side["name"].apply(normalize)
fpl_side["web_norm"] = fpl_side["web_name"].apply(normalize)

print(len(fpl_side))
fpl_side.head()

4131


,season,element,team,name,web_name,name_norm,web_norm
0,2021-22,286,Man Utd,Eric Bailly,Bailly,eric bailly,bailly
1,2021-22,49,Aston Villa,Keinan Davis,Davis,keinan davis,davis
2,2021-22,394,Watford,Ayotomiwa Dele-Bashiru,Dele-Bashiru,ayotomiwa dele bashiru,dele bashiru
3,2021-22,341,Southampton,James Ward-Prowse,Ward-Prowse,james ward prowse,ward prowse
4,2021-22,277,Man Utd,Bruno Miguel Borges Fernandes,Fernandes,bruno miguel borges fernandes,fernandes


In [10]:
# one row per player, season and club
tm_side = val_pl[["season", "team", "player_id"]].drop_duplicates()
tm_side = tm_side.merge(players[["player_id", "name"]], on="player_id", how="left")
tm_side["name_norm"] = tm_side["name"].apply(normalize)

print(len(tm_side))
tm_side.head()

3020


,season,team,player_id,name,name_norm
0,2021-22,Chelsea,102017,Jorginho,jorginho
1,2021-22,Watford,120186,Daniel Bachmann,daniel bachmann
2,2021-22,Everton,130164,Jordan Pickford,jordan pickford
3,2021-22,Spurs,167799,Pierre-Emile Højbjerg,pierre emile hojbjerg
4,2021-22,Man Utd,177907,Harry Maguire,harry maguire


In [11]:
# exact match on season + club + normalized name
exact = fpl_side.merge(tm_side, on=["season", "team", "name_norm"], how="inner", suffixes=("_fpl", "_tm"))
print(len(exact), "exact matches")

2396 exact matches


So far: 2396 exact matches out of 3020 Transfermarkt rows (79%).

For the rest, I'll use fuzzy matching by comparing names with **token_set_ratio** inside the same club and season. 

In [12]:
# Some examples I checked by hand: same player or not
examples = [
    ("mitoma kaoru", "kaoru mitoma", True),
    ("pedro lomba neto", "pedro neto", True),
    ("joseph willock", "joe willock", True),
    ("max kilman", "maximilian kilman", True),
    ("bruno fernandes", "bruno guimaraes", False),
    ("jordan ayew", "andre ayew", False),
    ("ben davies", "ben white", False),
]

rows = []
for fpl_name, tm_name, same_player in examples:
    rows.append({"fpl": fpl_name, "tm": tm_name, "same_player": same_player, "score": fuzz.token_set_ratio(fpl_name, tm_name)})

pd.DataFrame(rows)

,fpl,tm,same_player,score
0,mitoma kaoru,kaoru mitoma,True,100.000000
1,pedro lomba neto,pedro neto,True,100.000000
2,joseph willock,joe willock,True,88.000000
3,max kilman,maximilian kilman,True,75.000000
4,bruno fernandes,bruno guimaraes,False,66.666667
5,jordan ayew,andre ayew,False,66.666667
6,ben davies,ben white,False,63.157895


True matches go down to 75 (Kilman), false ones stay at 67 or less, so I set the threshold to 75. The gap is small, so maybe I'll have to check every fuzzy match under ~85 by hand.

For **web_name** I use **token_sort_ratio** instead: it's often just one word ("Gabriel"), and **token_set_ratio** gives 100 as soon as that word is inside a longer name ("Gabriel Jesus").

In [13]:
# FPL rows and TM rows not matched exactly yet
matched_fpl = exact[["season", "element", "team"]]
matched_tm = exact[["season", "team", "player_id"]]

fpl_left = fpl_side.merge(matched_fpl, how="left", indicator=True)
fpl_left = fpl_left[fpl_left["_merge"] == "left_only"].drop(columns="_merge")

tm_left = tm_side.merge(matched_tm, how="left", indicator=True)
tm_left = tm_left[tm_left["_merge"] == "left_only"].drop(columns="_merge")

LIMIT = 75

fuzzy_rows = []
for _, p in fpl_left.iterrows():
    # candidates: same season and club only
    candidates = tm_left[(tm_left["season"] == p["season"]) & (tm_left["team"] == p["team"])]
    best_id, best_score, best_name = None, 0, None
    for _, c in candidates.iterrows():
        # best of full name and web_name
        score = max(fuzz.token_set_ratio(p["name_norm"], c["name_norm"]),
                    fuzz.token_sort_ratio(p["web_norm"], c["name_norm"]))  # stricter for short names
        if score > best_score:
            best_id, best_score, best_name = c["player_id"], score, c["name"]
    if best_score >= LIMIT:
        fuzzy_rows.append({"season": p["season"], "element": p["element"], "team": p["team"],
                           "name_fpl": p["name"], "name_tm": best_name,
                           "player_id": best_id, "score": best_score})

fuzzy = pd.DataFrame(fuzzy_rows)
print(len(fuzzy), "fuzzy matches")
fuzzy.sort_values("score").head(20)  # lowest scores firstt to check these by hand (all correct, except the Gabriels)

504 fuzzy matches


,season,element,team,name_fpl,name_tm,player_id,score
43,2021-22,439,Wolves,Max Kilman,Maximilian Kilman,525247,75.000000
296,2024-25,524,West Ham,Max Kilman,Maximilian Kilman,525247,75.000000
47,2021-22,60,Brighton,Solomon March,Solly March,209212,75.000000
256,2023-24,563,Wolves,Max Kilman,Maximilian Kilman,525247,75.000000
62,2021-22,472,Watford,Juan Camilo Hernández Suárez,Cucho Hernández,459463,75.000000
113,2022-23,484,Wolves,Max Kilman,Maximilian Kilman,525247,75.000000
249,2023-24,294,Liverpool,Diogo Teixeira da Silva,Diogo Jota,340950,77.777778
355,2024-25,317,Liverpool,Diogo Teixeira da Silva,Diogo Jota,340950,77.777778
130,2022-23,10,Arsenal,Benjamin White,Ben White,335721,78.260870
465,2025-26,11,Arsenal,Benjamin White,Ben White,335721,78.260870


Sometimes the same Transfermarkt player gets picked for two FPL players in the same club and season (Martinelli and Magalhães both "matched" to "Gabriel"). I can't know which one is right automatically, so I drop those rows and fix them by hand below.

In [14]:
# a TM player given to two FPL players in the same club and season is very ambiguous
fix = fuzzy.duplicated(["season", "team", "player_id"], keep=False)
print(fix.sum(), "ambiguous rows")
print(fuzzy[fix][["season", "team", "name_fpl", "name_tm", "score"]])

fuzzy = fuzzy[~fix] # removing ambiguous rows

8 ambiguous rows
      season     team                          name_fpl  name_tm  score
52   2021-22  Arsenal                 Gabriel Magalhães  Gabriel  100.0
56   2021-22  Arsenal  Gabriel Teodoro Martinelli Silva  Gabriel  100.0
136  2022-23  Arsenal          Gabriel Martinelli Silva  Gabriel  100.0
162  2022-23  Arsenal      Gabriel dos Santos Magalhães  Gabriel  100.0
200  2023-24  Arsenal      Gabriel dos Santos Magalhães  Gabriel  100.0
217  2023-24  Arsenal          Gabriel Martinelli Silva  Gabriel  100.0
310  2024-25  Arsenal          Gabriel Martinelli Silva  Gabriel  100.0
333  2024-25  Arsenal      Gabriel dos Santos Magalhães  Gabriel  100.0


Manual fixes: Transfermarkt calls Magalhães just "Gabriel", so I set both Arsenal Gabriels by hand (ids checked on Transfermarkt).

I also added 4 players whose nickname is too far from their FPL name, even with web_name (score between 67 and 74, just under my limit): Kiko Femenía, Jonny Otto, Lucas Paquetá and Emerson Royal. I found them by looking at the regulars still missing at the end of this notebook.

In [15]:
# checked by hand
manual_fixes = {
    "Gabriel dos Santos Magalhães": 435338,
    "Gabriel Magalhães": 435338,
    "Gabriel Teodoro Martinelli Silva": 655488,
    "Gabriel Martinelli Silva": 655488,
    # nicknames too far from the FPL name (score 67-74)
    "Francisco Femenía Far": 76467,                       # Kiko Femenía
    "Jonathan Castro Otto": 175446,                       # Jonny Otto
    "Lucas Tolentino Coelho de Lima": 444523,             # Lucas Paquetá
    "Emerson Aparecido Leite de Souza Junior": 476344,    # Emerson Royal
    "Emerson Leite de Souza Junior": 476344,
}

manual = fpl_side[fpl_side["name"].isin(manual_fixes.keys())].copy()
manual["player_id"] = manual["name"].map(manual_fixes)
manual[["season", "team", "name", "player_id"]]

,season,team,name,player_id
61,2021-22,Watford,Francisco Femenía Far,76467
416,2021-22,Arsenal,Gabriel Magalhães,435338
423,2021-22,Wolves,Jonathan Castro Otto,175446
432,2021-22,Arsenal,Gabriel Teodoro Martinelli Silva,655488
590,2021-22,Spurs,Emerson Aparecido Leite de Souza Junior,476344
1040,2022-23,Spurs,Emerson Leite de Souza Junior,476344
1117,2022-23,Arsenal,Gabriel Martinelli Silva,655488
1292,2022-23,Wolves,Jonathan Castro Otto,175446
1307,2022-23,Arsenal,Gabriel dos Santos Magalhães,435338
1357,2022-23,West Ham,Lucas Tolentino Coelho de Lima,444523


## Step 6 — Final mapping

Putting exact, fuzzy and manual matches together.

In [16]:
cols = ["season", "element", "team", "player_id"]

# manual first, so it wins when a row appears twice
matches = pd.concat([manual[cols], exact[cols], fuzzy[cols]])
matches = matches.drop_duplicates(subset=["season", "element", "team"], keep="first")

print(len(matches), "matched rows")

2910 matched rows


In [17]:
# final table: one player_id per (season, element)
player_table = matches[["season", "element", "player_id"]].drop_duplicates()

n_ids = player_table.groupby(["season", "element"])["player_id"].nunique()
assert (n_ids == 1).all(), "an FPL player has two Transfermarkt ids"

print(len(player_table), "FPL players with a Transfermarkt id")

2862 FPL players with a Transfermarkt id


In [18]:
# minutes per FPL player and season
mins = gw.groupby(["season", "element"], as_index=False)["minutes"].sum()
regulars = mins[mins["minutes"] >= 450]

# which regulars got a Transfermarkt id
check = regulars.merge(player_table, on=["season", "element"], how="left", indicator=True)
rate = (check["_merge"] == "both").mean()

print(len(regulars), "players with 450+ min, match rate:", round(rate * 100, 1), "%")
# the ones still missing, with their names
missing = check[check["_merge"] == "left_only"]
missing.merge(fpl_side[["season", "element", "team", "name"]], on=["season", "element"])

2005 players with 450+ min, match rate: 99.6 %


,season,element,minutes,player_id,_merge,team,name
0,2022-23,661,653,NaN,left_only,Chelsea,Lewis Hall
1,2024-25,264,922,NaN,left_only,Ipswich,Wes Burns
2,2024-25,283,630,NaN,left_only,Ipswich,Christian Walton
3,2024-25,285,2165,NaN,left_only,Leicester,Boubakary Soumaré
4,2024-25,289,717,NaN,left_only,Leicester,Patson Daka
5,2024-25,301,1122,NaN,left_only,Leicester,Caleb Okoli
6,2024-25,304,900,NaN,left_only,Leicester,Jakub Stolarczyk
7,2024-25,764,1170,NaN,left_only,Ipswich,Alex Palmer


## Step 7 — How many valuations are left for the model

The match rate above is about FPL players. What really matters for the model is the other side: every Transfermarkt valuation is a training example, so how many of them got an FPL player?

In [19]:
# a valuation is kept if its (season, club, player) got an FPL match
kept = val_pl.merge(matches[["season", "team", "player_id"]].drop_duplicates(), how="left", indicator=True)
kept_rate = (kept["_merge"] == "both").mean()

print((kept["_merge"] == "both").sum(), "of", len(val_pl), "valuations kept:", round(kept_rate * 100, 1), "%")

6920 of 7068 valuations kept: 97.9 %


## Saving the mapping

The next notebooks only need this table, so I save it instead of redoing the matching every time. It goes in data/ (not on GitHub), so to get it back you just run this notebook.

In [20]:
import os

os.makedirs("../data/processed", exist_ok=True)
player_table.to_csv("../data/processed/player_map.csv", index=False)
print(len(player_table), "rows saved")

2862 rows saved


## Conclusion

In a nutshell, what I did: Premier League clubs per season from FPL, a season for each Transfermarkt value (cut on July 1st), only valuations of clubs that were in the PL that season, then names: exact match first, fuzzy match (token_set_ratio, limit 75) for the rest, and a few fixes by hand.

Results:
- 99.6% of FPL players with 450+ minutes have their Transfermarkt id.
- About 98% of Premier League valuations are kept for the model.
- The 8 players still missing (LLeicester and Ipswich players in 2024-25) have no Transfermarkt value at that club that season, so there is nothing to match.

At first I loaded only the Transfermarkt values tagged GB1 (English Premier League). When I looked at the last regulars still missing, Firmino and Pukki were in the list, and their values were actually there but got thrown away at loading. That column isn't the league of the club at the date of the value: Firmino's Liverpool values are tagged SA1 (Saudi league, where he plays now) and Pukki's Norwich ones are empty. And in 01 I had already seen the opposite (Championship clubs tagged GB1). So the column is wrong both ways, and I removed the filter. My own filter (club name + season, checked against the clubs FPL has) already does the job, and it got 76 Premier League valuations back, so worth it in my opinion.
